# GCNII on Protein-Protein Interaction (PPI) Dataset

Inductive Node Classification on PPI: Deep multi-layer GCNII for inductive multi-label protein interaction prediction. This notebook implements the approach with `GCN2Conv` inside a `K3GCNIIPPI` model, trained with the Adam optimizer for 10 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GCN2Conv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import PPI
from k3_node.loader import DataLoader

title = "GCNII on Protein-Protein Interaction (PPI) Graph"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Loaders
train_dataset = PPI(root="./data/PPI", split="train")
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

in_channels = train_dataset.num_features
out_channels = train_dataset.num_classes

# 2. GCNII PPI Model Definition
class K3GCNIIPPI(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=9, alpha=0.5, theta=1.0):
        super().__init__()
        self.lin1 = layers.Dense(hidden_channels)
        self.convs = [k3_layers.GCN2Conv(hidden_channels, alpha=alpha, theta=theta, layer=i + 1) for i in range(num_layers)]
        self.lin2 = layers.Dense(out_channels)

    def call(self, inputs, edge_index=None):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = ops.relu(self.lin1(x))
        x0 = x
        for conv in self.convs:
            x = ops.relu(conv(x, x0, edge_index))
        return self.lin2(x)

k3_model = K3GCNIIPPI(in_channels, 256, out_channels, num_layers=6)

# 3. Model Compilation (Multi-label BCE)
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(name="acc")],
)

# 4. Generator
def to_np(t, dtype=None):
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader):
    while True:
        for batch in loader:
            x = to_np(batch.x, dtype=np.float32)
            edge_index = to_np(batch.edge_index, dtype=np.int64)
            y = to_np(batch.y, dtype=np.float32)
            yield (x, edge_index), y

print(f"Training K3-Node GCNII PPI model on {backend} backend...")
history = k3_model.fit(
    make_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=10,
    verbose=1,
)

print("\n✓ K3-Node execution completed successfully!")